# Capítulo 27 · Laboratório 27B: o backtest em 10 passos
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**, na mesma cópia local do Laboratório 27 (`dados_cap27.pkl`). Se ela não existir, o notebook baixa os dados do Yahoo Finance e do Banco Central na sua máquina e grava essa cópia, que as execuções seguintes reaproveitam. O Yahoo corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. Uma regra pública e simples (rompimento de 50 dias, saída na mínima de 20) segue os 10 passos do Quadro do Capítulo 27, do mínimo de estatística à decisão. Custo de 0,10% ida e volta. O sorteio do Monte Carlo usa semente fixa: os números se repetem a cada execução.

In [ ]:
!pip install -q yfinance

In [ ]:
import io
import logging
import re
import os
import time
import urllib.request
import numpy as np
import pandas as pd
import yfinance as yf
from scipy import stats

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
B3 = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
      "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "CSAN3", "RAIL3", "CMIG4"]
EUA = ["AAPL", "MSFT", "JPM", "JNJ", "PG", "KO", "XOM", "CVX", "WMT", "HD", "MRK", "PFE", "CSCO", "INTC",
       "IBM", "MMM", "CAT", "DIS", "MCD", "BA"]
TICKERS = [t + ".SA" for t in B3] + EUA
FIM, CUSTO = "2026-10-01", 0.001
CACHE = "dados_cap27.pkl"


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


def mostra(*partes):
    """print com vírgula decimal e ponto de milhar (padrão brasileiro)."""
    texto = " ".join(str(p) for p in partes)
    texto = re.sub(r"(?<=\d)\.(?=\d)", ",", texto)
    print(re.sub(r"(?<=\d)_(?=\d)", ".", texto))


def limpa_estrita(d):
    """Tira barras vazias, preços não positivos e barras sem negócio (máxima igual à mínima ou volume zero)."""
    d = d.dropna()
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] > d["Low"]) & (d["Volume"] > 0)
    return d[ok]


def ajusta_saltos(d, ret_indice, limite=0.30):
    """Salto de mais de 30% num dia em que o índice andou menos de um quinto disso: trata como desdobramento ou
    grupamento não ajustado e corrige o histórico anterior pela razão do salto. Devolve a série corrigida e as datas.
    A regra é automática e erra para os dois lados: confira cada caso numa segunda fonte."""
    d = d.copy()
    r = d["Close"].pct_change()
    suspeitos = r[(r.abs() > limite) & (ret_indice.reindex(d.index).abs().fillna(0) < r.abs() / 5)]
    for dia, v in suspeitos.items():
        antes = d.index < dia
        d.loc[antes, ["Open", "High", "Low", "Close"]] *= 1 + v
    return d, list(suspeitos.items())

## Parte 1: o mínimo de estatística

In [ ]:
if os.path.exists(CACHE):              # a mesma cópia do Laboratório 27: o provedor corrige o histórico entre execuções
    dados, ib, cdi_bruto = pd.read_pickle(CACHE)
else:
    dados = yf.download(TICKERS, start="2005-01-01", end=FIM, auto_adjust=True, progress=False, group_by="ticker")
    ib = yf.download("^BVSP", start="1999-01-01", end=FIM, auto_adjust=True, progress=False, multi_level_index=False)
    confere({"as 40 ações": dados, "^BVSP": ib})
    url = ("https://api.bcb.gov.br/dados/serie/bcdata.sgs.4391/dados?formato=csv&dataInicial=01/01/1999"
           "&dataFinal=30/09/2026")
    for tentativa in range(5):                              # a API do Banco Central às vezes falha: tenta de novo
        try:
            cdi_bruto = urllib.request.urlopen(url, timeout=60).read().decode("latin-1")
            break
        except Exception:
            time.sleep(5 * (tentativa + 1))
    else:
        raise RuntimeError("Não consegui baixar o CDI do Banco Central depois de 5 tentativas. O site pode estar fora do"
                           " ar. Espere alguns minutos e rode a célula de novo.")
    pd.to_pickle((dados, ib, cdi_bruto), CACHE)
cdi_m = pd.read_csv(io.StringIO(cdi_bruto), sep=";", decimal=",")
cdi_m.index = pd.to_datetime(cdi_m["data"], dayfirst=True).dt.to_period("M")
cdi_m = cdi_m["valor"].astype(float) / 100
ibc = ib["Close"].dropna().loc["2000":]
dias_mes = ibc.groupby(ibc.index.to_period("M")).transform("size")
cdi_d = pd.Series((1 + cdi_m.reindex(ibc.index.to_period("M")).to_numpy()) ** (1 / dias_mes.to_numpy()) - 1,
                  index=ibc.index).fillna(0)

r = ibc.pct_change().dropna()
anos = len(r) / 252
media_a, vol_a = r.mean() * 252, r.std() * np.sqrt(252)
cagr = (ibc.iloc[-1] / ibc.iloc[0]) ** (1 / anos) - 1
mostra(f"Parte 1. O mínimo de estatística: Ibovespa, {r.index[0]:%d/%m/%Y} a {r.index[-1]:%d/%m/%Y} ({len(r):_} pregões)")
mostra(f"   retorno: média diária {r.mean():.3%} -> {media_a:.1%} ao ano (média aritmética);"
       f" composto (CAGR) {cagr:.1%} ao ano")
mostra(f"   volatilidade: desvio diário {r.std():.2%} -> {vol_a:.1%} ao ano (vezes a raiz de 252);"
       f" a diferença entre as duas médias é perto de metade da variância: {vol_a ** 2 / 2:.1%}")
z = (r - r.mean()) / r.std()
for k in (3, 4, 5):
    esperado = 2 * stats.norm.sf(k) * len(r)
    mostra(f"   caudas: dias além de {k} desvios: {int((z.abs() > k).sum())} observados x {esperado:.1f} numa curva normal")
mostra(f"   pior dia {r.min():.1%} em {r.idxmin():%d/%m/%Y} ({z.min():.1f} desvios); curtose em excesso {stats.kurtosis(r):.1f}"
       f" (0 na normal); assimetria {stats.skew(r):+.2f}")
R20 = pd.DataFrame({t: limpa_estrita(dados[t])["Close"].pct_change() for t in TICKERS[:20]}).loc["2016":]
corr_todas = R20.corr().to_numpy()[np.triu_indices(20, 1)].mean()
agitados = r.loc["2016":].abs().nlargest(60).index
corr_crise = R20.loc[R20.index.intersection(agitados)].corr().to_numpy()[np.triu_indices(20, 1)].mean()
mostra(f"   correlação média entre as 20 ações da B3 (2016 a 2026): {corr_todas:.2f} em todos os pregões;"
       f" {corr_crise:.2f} nos 60 pregões de maior oscilação do Ibovespa")
anual = ibc.resample("YE").last().pct_change().dropna()
anual = anual[anual.index.year <= 2025]
ep = anual.std() / np.sqrt(len(anual))
mostra(f"   erro-padrão: {len(anual)} anos completos, retorno médio {anual.mean():+.1%} ao ano, desvio {anual.std():.1%};"
       f" erro-padrão da média {ep:.1%}; intervalo de 95% de {anual.mean() - 1.96 * ep:+.1%} a {anual.mean() + 1.96 * ep:+.1%}")
mostra(f"   para estreitar o intervalo para ± 2 pontos ao ano seriam precisos cerca de"
       f" {int(np.ceil((1.96 * anual.std() / 0.02) ** 2)):_} anos de dados")

## Parte 2: passos 1 a 3: hipótese, dados e divisão da amostra

In [ ]:
ENTRA, SAI = 50, 20                       # passo 1: escritos antes de qualquer teste
CRITERIOS = {                             # passo 1: o que aprova a regra, também escrito antes
    "expectativa por operação acima de zero, fora da amostra": lambda m: m["expectativa (R)"] > 0,
    "índice de Sharpe acima de zero (excesso sobre o CDI), fora da amostra": lambda m: m["Sharpe"] > 0,
    "maior queda até 1,5 vez a de dentro da amostra": lambda m: m["maior queda"] >= 1.5 * DENTRO["maior queda"],
    "MAR fora pelo menos metade do MAR de dentro": lambda m: m["MAR"] >= 0.5 * DENTRO["MAR"],
}
CARTEIRA = TICKERS[:20]
D, saltos, sem_negocio = {}, {}, 0
ib_ret = ib["Close"].pct_change()
for t in TICKERS:
    base = limpa_estrita(dados[t])
    sem_negocio += len(dados[t].dropna()) - len(base)
    D[t], saltos[t] = ajusta_saltos(base, ib_ret if t.endswith(".SA") else pd.Series(dtype=float))
DENTRO_P, FORA_P = ("2006", "2015"), ("2016", "2026")
mostra(f"\nParte 2. Passo 1: a hipótese. Rompimento de {ENTRA} dias, só comprado; saída no fechamento abaixo da mínima"
       f" de {SAI} dias,\n   que é também o stop inicial: o risco de 1 R é a distância da entrada até essa mínima")
mostra("   critérios de aprovação, escritos antes do teste:")
for c in CRITERIOS:
    mostra(f"      - {c}")
mostra(f"   passo 2: os dados: {len(CARTEIRA)} ações da B3, peso igual entre as que têm negócio no dia (rebalanceado"
       " todo dia), preços ajustados por proventos; parte fora"
       " do mercado rende CDI;\n   são as ações de hoje (viés de sobrevivência)")
mostra(f"   limpeza: {sem_negocio:_} barras sem negócio ou sem volume registrado retiradas nas 40 ações")
for t, lista in saltos.items():
    for dia, v in lista:
        mostra(f"   salto suspeito ajustado: {t[:-3] if t.endswith('.SA') else t} em {dia:%d/%m/%Y}, {v:+.0%} num dia,"
               " com o Ibovespa quase parado")
mostra("   a regra automática erra para os dois lados: um salto pode ser real (notícia, ação de preço muito baixo)"
       " e um erro menor que 30% passa; confira numa segunda fonte")
mostra(f"   passo 3: dentro da amostra {DENTRO_P[0]} a {DENTRO_P[1]} (desenvolvimento); fora {FORA_P[0]} a 30/09/2026,"
       " guardado até o passo 10")

## Parte 3: passos 4 e 5: o sinal sem olhar o futuro e os custos

In [ ]:
def rompimento(d, entra=ENTRA, sai=SAI):
    """Posição (1 = comprado) decidida no fechamento de cada dia, e as operações com o resultado em % e em R."""
    c, h, l = d["Close"].to_numpy(), d["High"].to_numpy(), d["Low"].to_numpy()
    topo = pd.Series(h).rolling(entra).max().shift(1).to_numpy()      # máxima dos N dias anteriores (sem hoje)
    fundo = pd.Series(l).rolling(sai).min().shift(1).to_numpy()       # mínima dos M dias anteriores (sem hoje)
    pos, ops, dentro_, i0, stop0 = np.zeros(len(c)), [], False, 0, 0.0
    for i in range(max(entra, sai), len(c)):
        if not dentro_ and c[i] > topo[i]:
            dentro_, i0, stop0 = True, i, fundo[i]
        elif dentro_ and c[i] < fundo[i]:
            dentro_ = False
            risco = c[i0] - stop0
            ops.append({"entrada": d.index[i0], "saida": d.index[i], "r": c[i] / c[i0] - 1 - CUSTO,
                        "R": (c[i] - c[i0] - CUSTO * c[i0]) / risco, "risco": risco / c[i0], "dias": i - i0})
        pos[i] = 1.0 if dentro_ else 0.0
    if dentro_:                                   # a operação aberta no fim entra pelo último fechamento (viés da janela)
        i, risco = len(c) - 1, c[i0] - stop0
        ops.append({"entrada": d.index[i0], "saida": d.index[i], "r": c[i] / c[i0] - 1 - CUSTO,
                    "R": (c[i] - c[i0] - CUSTO * c[i0]) / risco, "risco": risco / c[i0], "dias": i - i0})
    return pd.Series(pos, index=d.index), pd.DataFrame(ops)


def diario(t, pos, atraso=1, custo=CUSTO, caixa=True):
    """Retorno diário de um ativo: posição de ontem (atraso=1) ou de hoje (atraso=0, olha o futuro)."""
    ret = D[t]["Close"].pct_change().fillna(0)
    p = pos.shift(atraso).fillna(0) if atraso else pos
    fora = cdi_d.reindex(ret.index).fillna(0) if caixa else 0.0
    return p * ret + (1 - p) * fora - p.diff().abs().fillna(0) * custo / 2


def carteira(tickers, atraso=1, custo=CUSTO, entra=ENTRA, sai=SAI, caixa=True):
    """Carteira de peso igual entre as ações com dados no dia; devolve o retorno diário, a exposição e as operações."""
    rets, exps, ops = {}, {}, []
    for t in tickers:
        pos, o = rompimento(D[t], entra, sai)
        rets[t] = diario(t, pos, atraso, custo, caixa)
        exps[t] = pos.shift(atraso).fillna(0) if atraso else pos
        ops.append(o.assign(ativo=t))
    return pd.DataFrame(rets).mean(axis=1), pd.DataFrame(exps).mean(axis=1), pd.concat(ops, ignore_index=True)


def anual(rd):
    eq = (1 + rd).cumprod()
    return eq.iloc[-1] ** (252 / len(rd)) - 1


per = slice(*DENTRO_P)
certo, expo, OPS = carteira(CARTEIRA)
futuro, _, _ = carteira(CARTEIRA, atraso=0)
sem_custo, _, _ = carteira(CARTEIRA, custo=0)
caro, _, _ = carteira(CARTEIRA, custo=0.006)
mostra(f"\nParte 3. Passo 4: o sinal sai no fechamento, a ordem é executada nesse fechamento e o retorno só conta a"
       " partir do pregão seguinte (shift de 1 dia)")
mostra(f"   dentro da amostra: com o shift {anual(certo.loc[per]):+.1%} ao ano | sem o shift (olha o futuro)"
       f" {anual(futuro.loc[per]):+.1%} ao ano")
mostra(f"   passo 5: custos: sem custo {anual(sem_custo.loc[per]):+.1%} | 0,10% ida e volta {anual(certo.loc[per]):+.1%}"
       f" | 0,60% ida e volta (derrapagem em ação pouco negociada) {anual(caro.loc[per]):+.1%} ao ano")

## Parte 4: passos 6 e 7: a curva e as métricas em quatro grupos

In [ ]:
def submerso(eq):
    """Maior período abaixo do topo anterior, em pregões, e a fração dos pregões abaixo do topo."""
    abaixo = (eq < eq.cummax()).to_numpy()
    maior = atual = 0
    for a in abaixo:
        atual = atual + 1 if a else 0
        maior = max(maior, atual)
    return maior, abaixo.mean()


def maior_seq(x):
    m = a = 0
    for v in x:
        a = a + 1 if v <= 0 else 0
        m = max(m, a)
    return m


def metricas(rd, ex, ops, ini, fim):
    rd, ex = rd.loc[ini:fim], ex.loc[ini:fim]
    o = ops[(ops["entrada"] >= ini) & (ops["entrada"] <= pd.Timestamp(fim) + pd.offsets.YearEnd(0))]  # pela entrada
    o = o.sort_values("saida")
    eq = (1 + rd).cumprod()
    exc = rd - cdi_d.reindex(rd.index).fillna(0)
    queda = (eq / eq.cummax() - 1).min()
    seco, frac = submerso(eq)
    g, p = o.loc[o["r"] > 0, "r"], o.loc[o["r"] <= 0, "r"]
    por_ano = rd.groupby(rd.index.year).apply(lambda x: (1 + x).prod() - 1)
    cdi_ano = cdi_d.reindex(rd.index).fillna(0).groupby(rd.index.year).apply(lambda x: (1 + x).prod() - 1)
    return {
        "CAGR": anual(rd), "exposição": ex.mean(),
        "volatilidade": rd.std() * np.sqrt(252), "maior queda": queda, "maior tempo submerso (anos)": seco / 252,
        "pregões abaixo do topo": frac,
        "Sharpe": np.sqrt(252) * exc.mean() / exc.std(),
        "Sortino": np.sqrt(252) * exc.mean() / np.sqrt((exc.clip(upper=0) ** 2).mean()),
        "MAR": anual(rd) / -queda,
        "operações": len(o), "acerto": len(g) / len(o), "payoff": g.mean() / -p.mean(),
        "fator de lucro": g.sum() / -p.sum(), "expectativa (R)": o["R"].mean(),
        "erro-padrão (R)": o["R"].std() / np.sqrt(len(o)),
        "anos acima do CDI": (por_ano > cdi_ano).mean(), "maior sequência de perdas": maior_seq(o["r"]),
    }


GRUPOS = {
    "Retorno": [("CAGR", "{:+.1%} ao ano"), ("exposição", "{:.0%} do capital, em média")],
    "Risco e retorno sobre o risco": [("volatilidade", "{:.1%} ao ano"), ("maior queda", "{:.0%}"),
                                      ("maior tempo submerso (anos)", "{:.1f} anos"),
                                      ("pregões abaixo do topo", "{:.0%}"), ("Sharpe", "{:.2f}"),
                                      ("Sortino", "{:.2f}"), ("MAR", "{:.2f}")],
    "Consistência": [("operações", "{:_}"), ("acerto", "{:.0%}"), ("payoff", "{:.2f}"), ("fator de lucro", "{:.2f}"),
                     ("expectativa (R)", "{:+.2f} R"), ("erro-padrão (R)", "± {:.2f} R"),
                     ("anos acima do CDI", "{:.0%}"), ("maior sequência de perdas", "{}")],
}


def relatorio(m, outra=None, rot=("", "")):
    for grupo, itens in GRUPOS.items():
        mostra(f"   {grupo}")
        for k, f in itens:
            extra = f"   | {rot[1]}: {f.format(outra[k])}" if outra is not None else ""
            mostra(f"      {k:<30} {rot[0]}{': ' if rot[0] else ''}{f.format(m[k])}{extra}")


DENTRO = metricas(certo, expo, OPS, *DENTRO_P)
mostra(f"\nParte 4. Passos 6 e 7: as métricas em quatro grupos, dentro da amostra ({DENTRO_P[0]} a {DENTRO_P[1]});"
       " o quarto grupo, estabilidade, é o passo 9")
relatorio(DENTRO)
o_d = OPS[(OPS["entrada"] >= DENTRO_P[0]) & (OPS["entrada"] <= "2015-12-31")]
mostra(f"   risco inicial mediano (1 R): {o_d['risco'].median():.1%} do preço de entrada; operações que perderam mais"
       f" de 1 R (o preço pulou o stop): {(o_d['R'] < -1).mean():.0%}")

## Parte 5: passo 8: a régua

In [ ]:
segura = pd.DataFrame({t: D[t]["Close"].pct_change() for t in CARTEIRA}).mean(axis=1).fillna(0)
mostra(f"\nParte 5. Passo 8: a régua, dentro da amostra")
for rot, rd in [("a regra", certo), ("comprar e segurar (peso igual)", segura), ("só CDI", cdi_d.reindex(certo.index).fillna(0)),
                ("Ibovespa", ibc.pct_change().reindex(certo.index).fillna(0))]:
    x = rd.loc[per]
    eq = (1 + x).cumprod()
    mostra(f"   {rot:<32} {anual(x):+.1%} ao ano | maior queda {(eq / eq.cummax() - 1).min():.0%}")

## Parte 6: passo 9: estabilidade

In [ ]:
GRADE = [(e, s) for e in (20, 30, 40, 50, 60, 80, 100) for s in (10, 15, 20, 30, 40) if s < e]
viz = {}
for e, s in GRADE:
    rd, ex, op = carteira(CARTEIRA, entra=e, sai=s)
    viz[(e, s)] = metricas(rd, ex, op, *DENTRO_P)
V = pd.DataFrame(viz).T
posto = int(V["MAR"].rank(ascending=False).loc[(ENTRA, SAI)])
mostra(f"\nParte 6. Passo 9: estabilidade, ainda só dentro da amostra")
mostra(f"   vizinhança: {len(GRADE)} combinações de entrada (20 a 100) e saída (10 a 40), a regra incluída; ela é a"
       f" {posto}ª de {len(GRADE)} pelo MAR")
mostra(f"   MAR de todas: mediana {V['MAR'].median():.2f}, de {V['MAR'].min():.2f} a {V['MAR'].max():.2f};"
       f" expectativa acima de zero em {(V['expectativa (R)'] > 0).mean():.0%} das combinações")
r_eua, ex_eua, op_eua = carteira(EUA, caixa=False)
seg_eua = pd.DataFrame({t: D[t]["Close"].pct_change() for t in EUA}).mean(axis=1).fillna(0)
m_eua = metricas(r_eua, ex_eua, op_eua, *DENTRO_P)
eq_s = (1 + seg_eua.loc[per]).cumprod()
mostra(f"   outros ativos: a mesma regra nas 20 ações americanas (fora = caixa parado): {m_eua['CAGR']:+.1%} ao ano,"
       f" maior queda {m_eua['maior queda']:.0%}, expectativa {m_eua['expectativa (R)']:+.2f} R |"
       f" comprar e segurar {anual(seg_eua.loc[per]):+.1%}, maior queda {(eq_s / eq_s.cummax() - 1).min():.0%}")
RISCO = 0.005                                   # 0,5% do capital por operação (Capítulo 29)
seq = o_d.sort_values("saida")["R"].to_numpy()


def pior_queda_R(x):
    eq = np.cumprod(1 + RISCO * x)
    return (eq / np.maximum.accumulate(eq) - 1).min()


rng = np.random.default_rng(27)                 # gerador próprio desta Parte, com semente fixa
mc = np.array([(pior_queda_R(s), maior_seq(s)) for s in (rng.permutation(seq) for _ in range(5000))])
mostra(f"   Monte Carlo: as {len(seq)} operações de dentro, uma de cada vez, arriscando {RISCO:.1%} do capital,"
       f" em 5_000 ordens:")
mostra(f"      ordem real: pior queda {pior_queda_R(seq):.0%}, {maior_seq(seq)} perdas seguidas | embaralhadas: pior queda"
       f" mediana {np.median(mc[:, 0]):.0%}, 5% piores {np.percentile(mc[:, 0], 5):.0%}; perdas seguidas mediana"
       f" {np.median(mc[:, 1]):.0f}, 5% piores {np.percentile(mc[:, 1], 95):.0f}")

## Parte 7: passo 10: fora da amostra, registro e decisão

In [ ]:
FORA = metricas(certo, expo, OPS, *FORA_P)
mostra(f"\nParte 7. Passo 10: fora da amostra ({FORA_P[0]} a 30/09/2026), consultado uma única vez")
relatorio(FORA, DENTRO, rot=("fora", "dentro"))
fx = slice(*FORA_P)
for rot, rd in [("comprar e segurar (peso igual)", segura), ("só CDI", cdi_d.reindex(certo.index).fillna(0))]:
    eq = (1 + rd.loc[fx]).cumprod()
    mostra(f"   régua fora: {rot:<30} {anual(rd.loc[fx]):+.1%} ao ano | maior queda {(eq / eq.cummax() - 1).min():.0%}")
mostra("   critérios escritos no passo 1:")
aprovados = 0
for c, teste in CRITERIOS.items():
    ok = bool(teste(FORA))
    aprovados += ok
    mostra(f"      {'passou' if ok else 'não passou'}: {c}")
n_testes = len(GRADE) + 1 + 2 + 1
mostra(f"   registro: {n_testes} configurações rodadas (a regra e {len(GRADE) - 1} vizinhas, a versão sem shift, 2"
       " versões de custo e as ações americanas); fora da amostra consultado 1 vez")
mostra(f"   decisão: {aprovados} de {len(CRITERIOS)} critérios; "
       + ("a regra segue para o teste com pouco dinheiro e acompanhamento (Capítulo 29)" if aprovados == len(CRITERIOS)
          else "a regra não segue como está; mudar e testar de novo no mesmo período fora já é sobreajuste"))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
for rd, rot in [(certo, "a regra"), (segura, "comprar e segurar"), (cdi_d.reindex(certo.index).fillna(0), "só CDI")]:
    ax.plot((1 + rd.loc["2006":]).cumprod(), label=rot)
ax.axvline(pd.Timestamp("2016-01-01"), ls=":", color="gray")
ax.set_yscale("log")
ax.legend()
plt.show()

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, as funções `rompimento` e `diario` da Parte 3. Mostre os dois cuidados com o tempo: o `.shift(1)` em `topo` e `fundo` (a máxima de 50 e a mínima de 20 dos dias anteriores, sem hoje) e o `atraso=1` em `diario` (o retorno só conta a partir do pregão seguinte). Mostre também como o stop inicial define o R e por que a operação aberta no fim entra pelo último fechamento." Confira com a Parte 3, que imprime, dentro da amostra, +13,4% ao ano com o shift e +29,1% sem ele.
2. **Adaptar:** "Calcule as métricas da Parte 4 para [ATIVO] sozinho, com `carteira` (uma lista só com [ATIVO]) e `metricas`, só dentro da amostra (`DENTRO_P`). Se [ATIVO] não estiver entre as 40 ações, baixe-o à parte, com as mesmas datas, passe por `limpa_estrita` e `ajusta_saltos`, como na Parte 2, e guarde em `D`. Não mude `ENTRA`, `SAI`, `CUSTO` nem os `CRITERIOS`, e não rode o fora da amostra: ele fica guardado até o passo 10. Qualquer conclusão sobre [ATIVO] precisa do teste fora da amostra (Capítulo 27)." Antes de olhar o ativo novo, confira se a Parte 4 continua mostrando, nas 20 ações, +13,4% ao ano, 283 operações e expectativa de +0,18 R, como na Tabela 27.3.

Mais pedidos deste capítulo (entender e aplicar ao seu caso) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 2, mude a regra para rompimento de 20 dias com saída na mínima de 10 antes de rodar o resto. Os critérios do passo 1 continuam aprovando?
2. Na Parte 3, troque o custo de 0,10% por 0,30%. Quanto do retorno anual sobra?
3. Na Parte 6, troque o risco por operação de 0,5% para 2%. O que acontece com a pior queda do Monte Carlo?
4. Escreva um quinto critério no passo 1 (por exemplo, bater comprar e segurar fora da amostra). A regra ainda passa?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.